[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_00/MFM_ch0_drawdowns_vix/MFM_ch0_drawdowns_vix.ipynb)

# MFM_ch0_drawdowns_vix

**Modelling Financial Markets (MFM) — Chapter 0: Financial Markets in 2026**

S&P 500 level and drawdown from the running peak, 2000-2026, with the troughs of the dot-com crash, the Global Financial Crisis, COVID-19 and the 2022 inflation shock; VIX history with its largest peaks and the share of days below 20 and above 30; the VIX against the realised S&P 500 volatility of the next 21 trading days and the gap between implied and realised variance.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_0'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Style and data loader

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def bottom_legend(fig, ncol=3, handles=None, labels=None, fontsize=8):
    """Legenda sub figura (in afara axelor), dupa tight_layout; save_fig o include (bbox_inches='tight')."""
    plt.tight_layout()
    if handles is None:
        handles, labels, seen = [], [], set()
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in seen and not l.startswith('_'):
                    handles.append(h); labels.append(l); seen.add(l)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False, fontsize=fontsize)


def log_axis(ax, ticks):
    """Axa logaritmica cu etichete lizibile (fara notatie stiintifica)."""
    ax.set_yscale('log')
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())


In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
LOCAL_DIRS = ['../../../data/market', '../../data/market', '../data/market', 'data/market']   # repo clonat
START, END = '2000-01-01', '2026-09-18'

# nume -> (sursa, simbol, camp)
SERIES = {
    'S&P 500': ('market', 'GSPC.INDX', 'close'),
    'Euro Stoxx 50': ('market', 'STOXX50E.INDX', 'close'),
    'Nikkei 225': ('market', 'N225.INDX', 'close'),
    'VIX': ('market', 'VIX.INDX', 'close'),
    'BET-TR': ('market', 'BETTR.INDX', 'close'),
    'BET': ('market', 'BET', 'close'),
    'BET-XT': ('market', 'BETXT.INDX', 'close'),
    'US Treasuries 20y+ (TLT)': ('market', 'TLT.US', 'adjusted_close'),
    'SPY': ('market', 'SPY.US', 'adjusted_close'),
    'RSP': ('market', 'RSP.US', 'adjusted_close'),
    'IBIT': ('market', 'IBIT.US', 'close'),
    'IBIT volume': ('market', 'IBIT.US', 'volume'),
    'Gold': ('market', 'XAUUSD.FOREX', 'close'),
    'EUR/USD': ('market', 'EURUSD.FOREX', 'close'),
    'EUR/RON (market file)': ('market', 'EURRON.FOREX', 'close'),
    'Bitcoin': ('market', 'BTC-USD.CC', 'close'),
    'Ethereum': ('market', 'ETH-USD.CC', 'close'),
    'Tether (USDT)': ('market', 'USDT-USD.CC', 'close'),
    'Romania 10y': ('market', 'RO10Y.GBOND', 'close'),
    'Germany 10y': ('market', 'DE10Y.GBOND', 'close'),
    'Banca Transilvania': ('market', 'TLV.RO', 'adjusted_close'),
    'OMV Petrom': ('market', 'SNP.RO', 'adjusted_close'),
    'BRD': ('market', 'BRD.RO', 'adjusted_close'),
    'Transgaz': ('market', 'TGN.RO', 'adjusted_close'),
    'Romgaz': ('market', 'SNG.RO', 'adjusted_close'),
    'Hidroelectrica': ('market', 'H2O.RO', 'adjusted_close'),
    'EUR/RON': ('bnr', 'EUR', None),
    'DGS10': ('fred', 'DGS10', None),
    'DGS2': ('fred', 'DGS2', None),
    'FEDFUNDS': ('fred', 'FEDFUNDS', None),
    'USD per EUR': ('fred', 'DEXUSEU', None),
    'JPY per USD': ('fred', 'DEXJPUS', None),
    'T10Y2Y': ('fred', 'T10Y2Y', None),
    'ETF equities': ('fred', 'BOGZ1LM563064100Q', None),
    'All equities': ('fred', 'BOGZ1LM893064105Q', None),
    'Stablecoins': ('defillama', 'stablecoincharts/all', None),
}
WEEKDAYS_ONLY = {'EUR/USD', 'Gold'}


def read_market(symbol):
    """Fisierul de date de piata: local (repo clonat), altfel din repo-ul GitHub."""
    for d in LOCAL_DIRS:
        path = os.path.join(d, f'{symbol}.csv')
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True)
    return pd.read_csv(REPO_RAW + f'{symbol}.csv', index_col='date', parse_dates=True)


def fetch_daily(symbol, source='market', field='close', start=START, end=END):
    """Punctul unic de acces la date: intoarce o pd.Series zilnica (fara NaN)."""
    if source == 'market':
        s = read_market(symbol)[field]
    elif source == 'fred':
        raw = urllib.request.urlopen(
            f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={symbol}', timeout=90).read().decode()
        s = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True, na_values='.').iloc[:, 0]
    elif source == 'defillama':
        req = urllib.request.Request(f'https://stablecoins.llama.fi/{symbol}',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        d = json.load(urllib.request.urlopen(req, timeout=90))
        s = pd.Series({pd.to_datetime(int(x['date']), unit='s'): x['totalCirculatingUSD'].get('peggedUSD', 0)
                       for x in d}) / 1e9                                   # miliarde USD
    elif source == 'bnr':
        # cursul oficial BNR (RON pentru o unitate de valuta), arhive XML anuale
        import re
        rows = []
        for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
            url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=90).read().decode()
            for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
                m = re.search(rf'<Rate currency="{symbol}">([\d.]+)</Rate>', body)
                if m:
                    rows.append((d, float(m.group(1))))
        s = pd.Series(dict(rows))
    else:
        raise ValueError(f'unknown source: {source}')
    s = pd.to_numeric(s, errors='coerce')
    s.index = pd.to_datetime(s.index)
    s.index.name = 'Date'
    return s.sort_index().loc[start:end].dropna()


def load(name, start=START, end=END):
    """Seria cu numele din SERIES (pentru aur si EUR/USD: doar zilele lucratoare)."""
    source, symbol, field = SERIES[name]
    s = fetch_daily(symbol, source, field, start, end).rename(name)
    if name in WEEKDAYS_ONLY:
        s = s[s.index.dayofweek < 5]
    return s


def load_panel(names, start=START, end=END):
    """Mai multe serii aliniate pe data (NaN acolo unde o piata nu tranzactioneaza)."""
    return pd.concat([load(n, start, end) for n in names], axis=1)


## Helper functions

In [ ]:
EPISODES = [('Dot-com', '2000-01-01', '2003-12-31'), ('Global Financial Crisis', '2007-06-01', '2010-12-31'), ('COVID-19', '2020-01-01', '2020-12-31'), ('2022 inflation shock', '2022-01-01', '2023-12-31')]

def drawdown(p):
    """Drawdown fata de maximul anterior: P_t / max_{s<=t} P_s - 1."""
    return p / p.cummax() - 1

## Data

In [ ]:
px = load_panel(['S&P 500', 'VIX'])

## Charts

In [ ]:
def fig_sp500_drawdowns():
    s = px['S&P 500'].dropna()
    dd = drawdown(s)
    fig, axes = plt.subplots(2, 1, figsize=(4.7, 3.3), sharex=True, gridspec_kw={'height_ratios': [1.1, 1]})
    axes[0].plot(s.index, s.values, color=MainBlue, lw=0.8)
    pk, rc = pd.Timestamp('2007-10-09'), s.loc['2009-03-10':][s.loc['2009-03-10':] >= s.loc['2007-10-09']].index[0]
    axes[0].plot([pk, rc], [s[pk], s[pk]], color=Amber, lw=2.2, solid_capstyle='butt')
    axes[0].annotate(f'{(rc - pk).days / 365.25:.1f} years to regain\nthe Oct 2007 peak', (rc, s[pk]), xytext=(4, -3),
                     textcoords='offset points', fontsize=7, va='top', color='black')
    log_axis(axes[0], [1000, 2000, 4000, 8000])
    axes[0].set_ylabel('S&P 500 (log)')
    axes[1].fill_between(dd.index, dd.values, 0, color=IDAred, alpha=0.35, lw=0)
    axes[1].plot(dd.index, dd.values, color=IDAred, lw=0.5)
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    axes[1].set_ylabel('Drawdown')
    res = {}
    for name, a, b in EPISODES:
        seg = dd.loc[a:b]
        t, v = seg.idxmin(), seg.min()
        res[name] = (t.date(), v)
        dx, ha = {'COVID-19': (-4, 'right'), '2022 inflation shock': (6, 'left')}.get(name, (0, 'center'))
        axes[1].annotate(f'{name}\n{v:.0%}', (t, v), xytext=(dx, -2), textcoords='offset points',
                         ha=ha, va='top', fontsize=7)
    axes[1].set_ylim(dd.min() * 1.45, 0.02)
    axes[0].set_title('S&P 500, 2000-2026: level (log) and drawdown', fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch0_sp500_drawdowns')
    return res

In [ ]:
def fig_vix_regimes():
    v = px['VIX'].dropna()
    fig, ax = plt.subplots(figsize=(6.6, 2.1))
    ax.plot(v.index, v.values, color=MainBlue, lw=0.5)
    ax.axhline(20, color=Amber, ls='--', lw=0.8)
    ax.axhline(30, color=IDAred, ls='--', lw=0.8)
    ax.text(1.005, 20 / (v.max() * 1.18), 'VIX = 20', fontsize=7, color=Amber, transform=ax.transAxes, va='center')
    ax.text(1.005, 30 / (v.max() * 1.18), 'VIX = 30', fontsize=7, color=IDAred, transform=ax.transAxes, va='center')
    # cele mai mari varfuri, separate prin cel putin 2 ani
    peaks = []
    for t in v.sort_values(ascending=False).index:
        if all(abs((t - p).days) > 730 for p in peaks):
            peaks.append(t)
        if len(peaks) == 5:
            break
    for t in peaks:
        ax.annotate(f'{t:%b %Y}\n{v.loc[t]:.0f}', (t, v.loc[t]), xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=7)
    ax.set_ylim(0, v.max() * 1.18)
    ax.set_ylabel('VIX (implied volatility, %)')
    share = {'<20': (v < 20).mean(), '20-30': ((v >= 20) & (v < 30)).mean(), '>=30': (v >= 30).mean()}
    ax.set_title(f'VIX 2000-2026: calm most of the time (<20 on {share["<20"]:.0%} of days), '
                 f'with short violent spikes', fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch0_vix_regimes')
    return {t.date(): round(v.loc[t], 2) for t in peaks}, share

In [ ]:
def fig_vix_vrp(h=21):
    """VIX (implicita, % pe an) vs volatilitatea realizata a S&P 500 in urmatoarele h = 21 de zile de tranzactionare:
    sqrt(252/h * suma r^2), fara ferestrele incomplete de la final; jos: varianta implicita minus cea realizata."""
    v = px['VIX'].dropna(); s = px['S&P 500'].dropna()
    r = np.log(s).diff()
    rv = np.sqrt((r ** 2)[::-1].rolling(h).sum()[::-1].shift(-1) * 252 / h) * 100
    j = pd.concat([v, rv], axis=1, keys=['vix', 'rv']).dropna()
    gap = (j['vix'] / 100) ** 2 - (j['rv'] / 100) ** 2
    fig, axes = plt.subplots(2, 1, figsize=(4.7, 3.3), sharex=True, gridspec_kw={'height_ratios': [1.2, 1]})
    axes[0].plot(j.index, j['rv'], color=Orange, lw=0.5, label='Realised volatility, next 21 trading days')
    axes[0].plot(j.index, j['vix'], color=MainBlue, lw=0.5, label='VIX (implied, next 30 calendar days)')
    axes[0].set_ylabel('% per year'); axes[0].set_ylim(0, 110)
    axes[1].fill_between(gap.index, gap.values, 0, where=gap >= 0, color=Forest, alpha=0.6, lw=0,
                         label='Implied > realised variance')
    axes[1].fill_between(gap.index, gap.values, 0, where=gap < 0, color=IDAred, alpha=0.6, lw=0,
                         label='Implied < realised variance')
    axes[1].axhline(0, color=Gray, lw=0.5)
    axes[1].set_ylabel('Variance gap'); axes[1].set_ylim(-0.5, 0.2)
    axes[0].set_title('VIX vs subsequent realised volatility, S&P 500', fontsize=9, loc='left')
    bottom_legend(fig, ncol=2, fontsize=7.2)
    save_fig('ch0_vix_vrp')
    return dict(n=len(j), first=j.index[0].date(), last=j.index[-1].date(), mean_vix=j['vix'].mean(),
                mean_rv=j['rv'].mean(), share_vix_above=(j['vix'] > j['rv']).mean(), mean_gap=gap.mean(),
                share_gap_pos=(gap > 0).mean())

## Run

In [ ]:
print(fig_sp500_drawdowns())
print(fig_vix_regimes())
print(fig_vix_vrp())

![ch0_sp500_drawdowns](./ch0_sp500_drawdowns.png)

![ch0_vix_regimes](./ch0_vix_regimes.png)

![ch0_vix_vrp](./ch0_vix_vrp.png)

Output: `ch0_sp500_drawdowns.pdf`, `ch0_vix_regimes.pdf`, `ch0_vix_vrp.pdf`